In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
emotion_df = pd.read_csv(r"C:\Users\LOQ\OneDrive\Desktop\NLP\data\emotions.csv")
violence_df = pd.read_csv(r"C:\Users\LOQ\OneDrive\Desktop\NLP\data\violence.csv")
hate_df = pd.read_csv(r"C:\Users\LOQ\OneDrive\Desktop\NLP\data\hatespeech.csv")

In [3]:
emotion_df.drop(columns = ['Unnamed: 0'],inplace=True)
violence_df.drop(columns = ['Tweet_ID'],inplace=True)
hate_df = hate_df[['tweet','class']]

In [4]:
emotion_df.columns, violence_df.columns, hate_df.columns

(Index(['text', 'label'], dtype='object'),
 Index(['tweet', 'type'], dtype='object'),
 Index(['tweet', 'class'], dtype='object'))

In [5]:
# Rename columns
violence_df.rename(columns={'tweet': 'text', 'type': 'label'}, inplace=True)
hate_df.rename(columns={'tweet': 'text', 'class': 'label'}, inplace=True)

In [6]:
emotion_df.shape, violence_df.shape, hate_df.shape

((416809, 2), (39650, 2), (24783, 2))

In [7]:
emotion_df['label'].value_counts()

label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64

In [8]:
violence_df['label'].value_counts()

label
sexual_violence                 32648
Physical_violence                5946
emotional_violence                651
economic_violence                 217
Harmful_Traditional_practice      188
Name: count, dtype: int64

In [9]:
# Label encoding for violence dataset
label_encoder = LabelEncoder()
violence_df['label'] = label_encoder.fit_transform(violence_df['label'])

In [10]:
violence_df['label'].value_counts()

label
4    32648
1     5946
3      651
2      217
0      188
Name: count, dtype: int64

In [11]:
hate_df['label'].value_counts()

label
1    19190
2     4163
0     1430
Name: count, dtype: int64

In [12]:
# Reset index
emotion_df.reset_index(drop=True, inplace=True)
violence_df.reset_index(drop=True, inplace=True)
hate_df.reset_index(drop=True, inplace=True)

# DATA AUGMENTATION

In [13]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# English → French
en_fr_model_name = "Helsinki-NLP/opus-mt-en-fr"
en_fr_tokenizer = MarianTokenizer.from_pretrained(en_fr_model_name)
en_fr_model = MarianMTModel.from_pretrained(en_fr_model_name).to(device)

# French → English
fr_en_model_name = "Helsinki-NLP/opus-mt-fr-en"
fr_en_tokenizer = MarianTokenizer.from_pretrained(fr_en_model_name)
fr_en_model = MarianMTModel.from_pretrained(fr_en_model_name).to(device)


In [14]:
import os
import pandas as pd

In [15]:

def back_translate_batch(texts, batch_size=16):
    augmented_texts = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Back-translating"):
        batch_texts = texts[i:i + batch_size]

        # EN → FR
        en_inputs = en_fr_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        fr_outputs = en_fr_model.generate(**en_inputs)
        fr_texts = en_fr_tokenizer.batch_decode(
            fr_outputs, skip_special_tokens=True
        )

        # FR → EN
        fr_inputs = fr_en_tokenizer(
            fr_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        en_outputs = fr_en_model.generate(**fr_inputs)
        augmented_texts.extend(
            fr_en_tokenizer.batch_decode(en_outputs, skip_special_tokens=True)
        )

    return augmented_texts

In [21]:
surprise_df = emotion_df[emotion_df["label"] == 5]
texts = surprise_df["text"].tolist()

print(f"Surprise samples: {len(texts)}")

Surprise samples: 14972


In [22]:
torch.cuda.empty_cache()

augmented_texts = back_translate_batch(
    texts,
    batch_size=16   # Safe for RTX 4060 Laptop
)

Back-translating: 100%|██████████| 936/936 [17:04<00:00,  1.09s/it] 


In [23]:
augmented_df = pd.DataFrame({
    "text": augmented_texts,
    "label": 5
})

In [24]:
emotion_df_augmented = pd.concat(
    [emotion_df, augmented_df],
    ignore_index=True
)

In [25]:
print("\nClass distribution after augmentation:")
print(emotion_df_augmented["label"].value_counts())


Class distribution after augmentation:
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     29944
Name: count, dtype: int64


In [26]:
# =========================
# 9. Save CSV to your folder
# =========================
output_dir = r"C:\Users\LOQ\OneDrive\Desktop\NLP\data"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "emotion_augmented_backtranslation.csv"
)

emotion_df_augmented.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("\n✅ Augmentation complete!")
print("Saved at:", output_path)



✅ Augmentation complete!
Saved at: C:\Users\LOQ\OneDrive\Desktop\NLP\data\emotion_augmented_backtranslation.csv


In [27]:
TARGET_COUNTS = {
    0: 12000,  # Harmful traditional practice
    1: 18000,  # Physical violence
    2: 12000,  # Economic violence
    3: 12000,  # Emotional violence
    4: None    # Sexual violence (NO augmentation)
}


In [28]:
augmented_rows = []

for label, target_size in TARGET_COUNTS.items():

    class_df = violence_df[violence_df["label"] == label]
    current_size = len(class_df)

    print(f"\nClass {label}: {current_size} samples")

    # Skip majority class
    if target_size is None or current_size >= target_size:
        print(" → No augmentation needed")
        continue

    needed = target_size - current_size
    print(f" → Generating {needed} augmented samples")

    texts = class_df["text"].tolist()

    # Repeat texts cyclically if very small
    repeat_factor = (needed // len(texts)) + 1
    texts_expanded = (texts * repeat_factor)[:needed]

    torch.cuda.empty_cache()

    augmented_texts = back_translate_batch(
        texts_expanded,
        batch_size=16
    )

    for txt in augmented_texts:
        augmented_rows.append({
            "text": txt,
            "label": label
        })



Class 0: 188 samples
 → Generating 11812 augmented samples


Back-translating: 100%|██████████| 739/739 [23:47<00:00,  1.93s/it]



Class 1: 5946 samples
 → Generating 12054 augmented samples


Back-translating: 100%|██████████| 754/754 [36:07<00:00,  2.87s/it]  



Class 2: 217 samples
 → Generating 11783 augmented samples


Back-translating: 100%|██████████| 737/737 [35:27<00:00,  2.89s/it]  



Class 3: 651 samples
 → Generating 11349 augmented samples


Back-translating: 100%|██████████| 710/710 [23:58<00:00,  2.03s/it]  


Class 4: 32648 samples
 → No augmentation needed


In [29]:
augmented_df = pd.DataFrame(augmented_rows)


In [30]:
violence_df_augmented = pd.concat(
    [violence_df, augmented_df],
    ignore_index=True
)


In [31]:
print("\nClass distribution after augmentation:")
print(violence_df_augmented["label"].value_counts())


Class distribution after augmentation:
label
4    32648
1    18000
3    12000
0    12000
2    12000
Name: count, dtype: int64


In [32]:
output_dir = r"C:\Users\LOQ\OneDrive\Desktop\NLP\data"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "violence_augmented_backtranslation.csv"
)

violence_df_augmented.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

In [17]:
# =========================
# Hate augmentation ONLY
# =========================

TARGET_COUNTS_HATE = {
    0: 9000,   # Hate speech
    2: 9000,   # Neither
    1: None    # Offensive speech (NO augmentation)
}

augmented_rows = []

for label, target_size in TARGET_COUNTS_HATE.items():

    class_df = hate_df[hate_df["label"] == label]
    current_size = len(class_df)

    print(f"\nClass {label}: {current_size} samples")

    if target_size is None or current_size >= target_size:
        print(" → No augmentation needed")
        continue

    needed = target_size - current_size
    print(f" → Generating {needed} augmented samples")

    texts = class_df["text"].tolist()

    repeat_factor = (needed // len(texts)) + 1
    texts_expanded = (texts * repeat_factor)[:needed]

    torch.cuda.empty_cache()

    augmented_texts = back_translate_batch(
        texts_expanded,
        batch_size=16  # Hate texts are usually short
    )

    for txt in augmented_texts:
        augmented_rows.append({
            "text": txt,
            "label": label
        })



Class 0: 1430 samples
 → Generating 7570 augmented samples


Back-translating: 100%|██████████| 474/474 [27:34<00:00,  3.49s/it]  



Class 2: 4163 samples
 → Generating 4837 augmented samples


Back-translating: 100%|██████████| 303/303 [13:14<00:00,  2.62s/it] 


Class 1: 19190 samples
 → No augmentation needed


In [19]:
augmented_df = pd.DataFrame(augmented_rows)

hate_df_augmented = pd.concat(
    [hate_df, augmented_df],
    ignore_index=True
)




In [20]:

hate_df_augmented.to_csv(
    r"C:\Users\LOQ\OneDrive\Desktop\NLP\data\hate_augmented_backtranslation.csv",
    index=False,
    encoding="utf-8"
)

In [21]:

print(hate_df_augmented["label"].value_counts())

label
1    19190
2     9000
0     9000
Name: count, dtype: int64
